In [ ]:
%load_ext autoreload
%autoreload 2

from setup_imports import *  # noqa: F401,F403

# Tag phrases from text, then sync tags to Anki

Walks through the real end-to-end workflow with one concrete example:

1. `add_tags_from_text` — from a piece of target-language text, find the minimum set of *existing* phrases whose translation already covers its vocab, and tag those phrases in Firestore (tag stored **unprefixed**).
2. Confirm the tag landed on the phrase's translation in Firestore.
3. `sync_tag_to_anki` — sync that tag into the live Anki collection (`dry_run=True` first). It should show up on the note as `fs::food_and_drink` — the `fs::` prefix is added only at this step; Firestore itself keeps the bare `food_and_drink`.
4. Only once you're happy with the dry-run report: flip `RUN_FOR_REAL` to `True` and re-run the last cell to actually write to Anki.

In [ ]:
from phrases.search import add_tags_from_text, find_phrases_by_tag
from connections.anki_collection import get_anki_collection, close_anki_collection
from anki_sync import sync_tag_to_anki
from phrases.generation import generate_phrases_from_vocab_dict
from phrases.phrase_model import Phrase
from langcodes import Language
from anki_tools import create_anki_deck

## Config

In [ ]:
TITLE = "blood_sacrifice_s1_e1"

In [ ]:
# load some text

with open(f"../data/text_to_process/{TITLE}.txt", "r", encoding="utf-8") as f:
    text = f.readlines()

text = " ".join([line.strip() for line in text if line.strip()])

In [ ]:
text

In [ ]:
TEXT = text
# create an anki deck and import.
TARGET_LANGUAGE = Language.get("sv-SE")
print(TARGET_LANGUAGE.display_name())
SOURCE_LANGUAGE = Language.get("en-GB")
print(SOURCE_LANGUAGE.display_name())
TAG = f"tv_series::{TITLE}"

# Safety gate for the last cell - real Anki write only happens if True
RUN_FOR_REAL = True

## 1. Tag the covering phrase(s) in Firestore

Finds the minimum set of existing phrases whose Swedish translation covers the vocab in `TEXT`, and tags them. This is a real Firestore write (low-risk/reversible - see `delete_tag_from_firestore` in `phrases/search.py` if you need to undo it).

In [ ]:
TEXT

In [ ]:
tagged_phrases, missing = add_tags_from_text(
    text=TEXT,
    target_language=TARGET_LANGUAGE,
    tags=TAG,
    dry_run=False,
    min_occurrences=3,
)

In [ ]:
missing["vocab"]

In [ ]:
# generate missing phrases

In [ ]:
new_phrases = generate_phrases_from_vocab_dict(missing, TARGET_LANGUAGE)

In [ ]:
new_phrases = [
    "Jag dejtar honom nu",
    "Vi dejtade förra året",
    "Ska du dejta henne?",
    "Hon fantiserar om framtiden",
    "Jag fantiserade om dig",
    "Vi kommer att fantisera",
    "Jag funderar på det",
    "Hon funderade länge igår",
    "Ska du fundera mer?",
    "Vi ankrar båten nu",
    "De förankrade idén väl",
    "Ska du förankra beslutet?",
    "Hon förankrar förslaget brett",
    "Jag förtöjde skeppet igår",
    "Barnet gråter varje dag",
    "Jag grät igår kväll",
    "Kommer du att gråta?",
    "Himla inte med ögonen åt mig",
    "Hon himlade med ögonen",
    "Jag kommer att himla med ögonen",
    "Himla vackert väder idag",
    "Jag hinner inte idag",
    "Hann du med bussen?",
    "Vi hinner inte imorgon",
    "Hinner du läsa boken?",
    "Hon hann ikapp mig",
    "De knullar varje helg",
    "Vi knullade igår kväll",
    "Ska du knulla ikväll?",
    "Det känns bra idag",
    "Det kändes konstigt igår",
    "Hur kommer det att kännas?",
    "Tyget känns mjukt",
    "Varför ljuger du alltid?",
    "Han ljög om sitt jobb",
    "Jag kommer inte att ljuga",
    "Det luktar gott här",
    "Blommorna luktade underbart igår",
    "Jag ska lukta på den",
    "Lukta på mjölken först",
    "Maten luktar illa",
    "Jag lånar en bok",
    "Hon lånade min cykel",
    "Kan jag låna pengar?",
    "Jag lånar ut min bil",
    "De pippar hela natten",
    "Hon pippade mig",
    "Ska du pippa honom?",
    "Jag skriver ett brev",
    "Hon skrev en bok",
    "Vi ska skriva imorgon",
    "Jag vågar inte hoppa",
    "Vågade du fråga henne?",
    "Vi kommer att våga",
    "alltså en bra idé",
    "den bruna lappen",
    "en bög på redaktionen",
    "en dum journalist",
    "ett dumt experiment",
    "ensam på resmålet",
    "exakt det läget",
    "hennes exman och pappa",
    "fan vilken skit",
    "förankra idén nånstans",
    "förstås är det okej",
    "vilket helvete",
    "herpes som munsår",
    "en heterosexuell kvinnlig journalist",
    "inte hinna med lånet",
    "en härlig sak",
    "en jättebra idé",
    "igång med experimentet",
    "en jobbig lesbisk exfru",
    "det är ju kallt",
    "just den här tjejen",
    "en jättekul överraskning",
    "något kul utomlands",
    "den gula lappen",
    "ett bra läge",
    "hans nya lån",
    "min trasiga mobil",
    "den nakna mannen",
    "två nakna fötter",
    "nåt i väskan",
    "redaktionen på våningen",
    "ett exotiskt resmål",
    "för hennes skull",
    "tre hårda slag",
    "de vita sockerpillren",
    "det kostar femtio spänn",
    "ett svart streck",
    "den sura mjölken",
    "platsen är tagen",
    "ett kreativt uppslag",
    "även hans syster",
]

In [ ]:
# create the phrases - batched: translates and refines all phrases in 2 API calls
# instead of one Google Translate + one Claude call per phrase
ALL_NEW_PHRASES = Phrase.create_from_foreign_batch(
    new_phrases, TARGET_LANGUAGE, split_on_space=True, tags=[TAG]
)

In [ ]:
for phrase in ALL_NEW_PHRASES:
    sv_text = phrase.translations[TARGET_LANGUAGE.to_tag()].text
    print(f"{phrase.english} |  {sv_text}")

In [ ]:
for p in ALL_NEW_PHRASES:
    p.generate_audio(context="flashcard", language=TARGET_LANGUAGE, split_on_space=True)
    p.upload(language=TARGET_LANGUAGE)
    p.generate_image()
    p.upload()

## 2. Confirm the tag in Firestore, unprefixed

In [ ]:
P2 = find_phrases_by_tag(TAG, TARGET_LANGUAGE)

In [ ]:
ALL_NEW_PHRASES = sorted(
    ALL_NEW_PHRASES, key=lambda x: len(x.translations[str(TARGET_LANGUAGE)].text)
)

In [ ]:
# DON'T CREATE ANKI DECK IF PHRASE EXISTS ALREADY IN ANKI

create_anki_deck(
    ALL_NEW_PHRASES,
    source_language=SOURCE_LANGUAGE,
    target_language=TARGET_LANGUAGE,
    output_path=f"../outputs/decks/{TARGET_LANGUAGE.to_tag()}/{TARGET_LANGUAGE.to_tag()}-{TAG.replace('::', '-')}.apkg",
    deck_name=f"FirePhrase - {TARGET_LANGUAGE.language_name()}::{TAG}",
)

## 3. Sync the tag into the live Anki collection (dry run)

`dry_run=True` makes zero writes anywhere - safe to re-run as many times as you like.

In [ ]:
col = get_anki_collection()
try:
    report = sync_tag_to_anki(col, TAG, SOURCE_LANGUAGE, TARGET_LANGUAGE, dry_run=False)
    print(report.summary())
    for r in report.results:
        print(" ", r)
finally:
    close_anki_collection()

## 4. Run for real — only after reviewing the dry-run report above

Close Anki Desktop first (it holds an exclusive lock on the collection file). Set `RUN_FOR_REAL = True` in the config cell and re-run it, then run this cell. Takes a real Anki backup before writing, same as `scripts/sync_anki_tags.py`.

In [ ]:
import os
from pathlib import Path

if not RUN_FOR_REAL:
    print(
        "Skipped - set RUN_FOR_REAL = True in the config cell above and re-run both cells when ready."
    )
else:
    col = get_anki_collection()
    try:
        backup_folder = str(Path(os.environ["ANKI_COLLECTION_PATH"]).parent / "backups")
        os.makedirs(backup_folder, exist_ok=True)
        col.create_backup(
            backup_folder=backup_folder, force=True, wait_for_completion=True
        )

        report = sync_tag_to_anki(col, TAG, SOURCE_LANGUAGE, LANGUAGE, dry_run=False)
        print(report.summary())
        for r in report.results:
            print(" ", r)
    finally:
        close_anki_collection()